# Assignment 2 COMP 6934 W26

*Provincial Economy and Voting Patterns in the 2025 Canadian General Election*

## Overview / Abstract

This notebook explores the relationship between provincial economic structure and voting patterns in the 2025 Canadian General Election (45th Parliament). I apply Tamara Munzner's visualization analysis framework (what/why/how) to design and produce two original visualizations.

**Research Questions:**
1. *"Is there a relationship between a province's economics and its party support?"*
2. *"Is there a relationship between Alberta's conservative orientation and its industry, particularly its oil industry, as compared to other provinces?"*

**Visualization 1** is a pie-glyph scatter plot that positions each province by its economic profile (goods-to-services GDP ratio) against its Conservative vote share, with miniature pie charts encoding the full party seat distribution at each point.

**Visualization 2** is a dual-panel figure pairing normalized industry-composition bars with a lollipop chart of Conservative vote share, sorted by political outcome, with Alberta highlighted as the focal province.

### Reproduction Instructions
Run all cells sequentially from top to bottom. Required packages: `pandas`, `numpy`, `plotly`, `matplotlib` (used only for rendering pie-glyph images).

Using three datasets 
- Elections Canada riding-level results (`table_tableau12.csv`)
- House of Commons membership (`commons.csv`)
- Statistics Canada GDP-by-industry figures (`36100711-eng/36100711.csv`)


# General Description of Data Sets

### Dataset 1: Elections Canada - Riding-Level Voting Results (Table 12)

**Subject matter:** Official candidate-level voting results from the 45th Canadian General Election held on April 28, 2025, covering all 343 federal electoral districts (ridings) across 13 provinces and territories.

**Sourcing and collection:** Published by Elections Canada, the independent, non-partisan agency responsible for administering federal elections. Data is collected through the official electoral process at each polling station and aggregated to the riding le#vel. Downloaded from [Elections Canada](https://www.elections.ca/content.aspx?section=res&dir=rep/off/45gedata&document=summary&lang=e), specifically Table 12 (Candidates and Votes).

**Organization / format:** A flat CSV file (`table_tableau12.csv`) with 1,959 rows (one per candidate) and 10 columns with bilingual headers. Each row records a candidate's province, electoral district, name (with party affiliation embedded in the same field), residence, occupation, votes obtained, and vote percentage. Winners are identified by a non-null Majority column. Party affiliation is not in its own column - it is concatenated with the candidate's name (e.g., `"Paul Connors Liberal/Libéral"`).

### Dataset 2: House of Commons Members

**Subject matter:** Current Members of Parliament (MPs) sitting in the Canadian House of Commons, reflecting the results of the 2025 General Election plus any subsequent by-elections.

**Sourcing and collection:** Published by the House of Commons of Canada. The data is administratively maintained by the House of Commons. Downloaded as a CSV from [ourcommons.ca](https://www.ourcommons.ca/Members/en/search/csv).

**Organization / format:** A flat CSV file (`commons.csv`) with approximately 343 rows (one per MP) and 9 columns. Unlike the elections data, Political Affiliation has its own dedicated column. Fields include Person ID, First/Last Name, Constituency, Province/Territory, Political Affiliation, and Start/End Dates.

### Dataset 3: Statistics Canada - GDP by Industry, by Province (Table 36-10-0711-01)

**Subject matter:** Gross domestic product (GDP) at basic prices, broken down by industry (NAICS classification) and by province/territory, providing a comprehensive picture of each province's economic structure.

**Sourcing and collection:** Published by Statistics Canada as part of the national economic accounts program. Values are derived from administrative and survey data collected by Statistics Canada. Downloaded from [Statistics Canada](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=3610071101) as a full CSV download.

**Organization / format:** A large CSV file (`36100711.csv`) with approximately 269,000 rows and 16 columns. Each row represents a single observation: one province, one year (1997-2024), one NAICS industry classification, with a GDP value in millions of dollars. The NAICS codes form a hierarchy (e.g., "All industries [T001]" decomposes into "Goods-producing [T002]" and "Services-producing [T003]", which further decompose into specific sectors like "Oil and gas extraction [211]"). Data is available in both current and chained (2017) dollars.

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load all three datasets
elections = pd.read_csv('table_tableau12.csv', encoding='utf-8-sig')
commons = pd.read_csv('commons.csv', encoding='utf-8-sig')
statcan = pd.read_csv('36100711-eng/36100711.csv',
                      encoding='utf-8-sig', low_memory=False)

print(
    f"Elections data: {elections.shape[0]} rows x {elections.shape[1]} columns")
print(f"Commons data:   {commons.shape[0]} rows x {commons.shape[1]} columns")
print(f"StatCan data:   {statcan.shape[0]} rows x {statcan.shape[1]} columns")

Elections data: 1959 rows x 10 columns
Commons data:   340 rows x 9 columns
StatCan data:   269724 rows x 16 columns


In [51]:
elections.head()

,Province,Electoral District Name/Nom de circonscription,Electoral District Number/Numéro de circonscription,Candidate/Candidat,Candidate Residence/Résidence du candidat,Candidate Occupation/Profession du candidat,Votes Obtained/Votes obtenus,Percentage of Votes Obtained /Pourcentage des votes obtenus,Majority/Majorité,Majority Percentage/Pourcentage de majorité,Party,Is_Incumbent,Province_Clean
0,Newfoundland and Labrador/Terre-Neuve-et-Labrador,Avalon,10001,Paul Connors Liberal/Libéral,"Conception Bay South, N.L./ T.-N.-L.",Executive Assistant/Adjoint de direction,27563,58.6,10610.0,22.6,Liberal,False,Newfoundland and Labrador
1,Newfoundland and Labrador/Terre-Neuve-et-Labrador,Avalon,10001,Steve Kent Conservative/Conservateur,"Mount Pearl, N.L./ T.-N.-L.",Consultant/Consultant,16953,36.0,NaN,NaN,Conservative,False,Newfoundland and Labrador
2,Newfoundland and Labrador/Terre-Neuve-et-Labrador,Avalon,10001,Judy Vanta NDP-New Democratic Party/NPD-Nouvea...,"St. John's, N.L./ T.-N.-L.",Retired/À la retraite,2284,4.9,NaN,NaN,NDP,False,Newfoundland and Labrador
3,Newfoundland and Labrador/Terre-Neuve-et-Labrador,Avalon,10001,Alexander Tilley Parti Rhinocéros Party/Parti ...,"Holyrood, N.L./ T.-N.-L.",Seafood Worker/Ouvrier des fruits de mer,230,0.5,NaN,NaN,Rhinocéros,False,Newfoundland and Labrador
4,Newfoundland and Labrador/Terre-Neuve-et-Labrador,Cape Spear,10002,Tom Osborne Liberal/Libéral,"St. John's, N.L./ T.-N.-L.",General Manager/Directeur général,31388,68.3,19544.0,42.5,Liberal,False,Newfoundland and Labrador


In [52]:
commons.head()

,Person ID,Honorific Title,First Name,Last Name,Constituency,Province / Territory,Political Affiliation,Start Date,End Date
0,89156,NaN,Ziad,Aboultaif,Edmonton Manning,Alberta,Conservative,2025-04-28 12:00:00 a.m.,NaN
1,123092,NaN,Sima,Acan,Oakville West,Ontario,Liberal,2025-04-28 12:00:00 a.m.,NaN
2,105340,NaN,Scott,Aitchison,Parry Sound—Muskoka,Ontario,Conservative,2025-04-28 12:00:00 a.m.,NaN
3,123033,NaN,Fares,Al Soud,Mississauga Centre,Ontario,Liberal,2025-04-28 12:00:00 a.m.,NaN
4,72029,NaN,Dan,Albas,Okanagan Lake West—South Kelowna,British Columbia,Conservative,2025-04-28 12:00:00 a.m.,NaN


In [53]:
statcan.head()

,REF_DATE,GEO,DGUID,Prices,North American Industry Classification System (NAICS),UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,1997,Newfoundland and Labrador,2021A000210,Current dollars,All industries [T001],Dollars,81,millions,6,v1793281794,1.1.1,9588.4,NaN,NaN,NaN,1
1,1997,Newfoundland and Labrador,2021A000210,Current dollars,Goods-producing industries [T002],Dollars,81,millions,6,v1793281795,1.1.2,2364.5,NaN,NaN,NaN,1
2,1997,Newfoundland and Labrador,2021A000210,Current dollars,Services-producing industries [T003],Dollars,81,millions,6,v1793281796,1.1.3,7223.9,NaN,NaN,NaN,1
3,1997,Newfoundland and Labrador,2021A000210,Current dollars,Industrial production [T010],Dollars,81,millions,6,v1793281797,1.1.4,1386.7,NaN,NaN,NaN,1
4,1997,Newfoundland and Labrador,2021A000210,Current dollars,Non-durable manufacturing industries [T011],Dollars,81,millions,6,v1793281798,1.1.5,488.8,NaN,NaN,NaN,1


# Data Details - Munzner "What" Analysis

### Dataset Types and Structures

Following Munzner's data abstraction taxonomy:

| Dataset | Munzner Dataset Type | Structure |
|-----|-----------|------|
| Elections (Table 12) | **Table** | Flat table; each item (row) is one candidate in one riding |
| House of Commons | **Table** | Flat table; each item (row) is one MP |
| StatCan GDP | **Multidimensional Table** | Each item is a province x year x industry combination; the NAICS codes also form a **tree** (hierarchical) structure |
| Merged (derived for visualization) | **Table** | Province-level aggregation linking economic and political attributes |

### Attribute Analysis

#### Elections Canada (Table 12)

| Attribute | Munzner Attribute Type | Ordering |
|------|-----------|-----|
| Province | Categorical | Unordered |
| Electoral District Name | Categorical (identifier) | Unordered |
| Electoral District Number | Categorical (identifier) | Unordered |
| Candidate Name (extracted) | Categorical | Unordered |
| Party (extracted from candidate field) | Categorical | Unordered |
| Candidate Residence | Categorical | Unordered |
| Candidate Occupation | Categorical | Unordered |
| Votes Obtained | Quantitative | Sequential (ratio scale; zero is meaningful) |
| Percentage of Votes Obtained | Quantitative | Sequential (ratio; range 0-100) |
| Majority | Quantitative | Sequential (ratio; present only for winners) |
| Majority Percentage | Quantitative | Sequential (ratio) |

**Observations:** The party affiliation is embedded in the Candidate field and must be extracted through string parsing. The `**` marker indicates incumbent candidates. The Majority field being non-null serves as an implicit boolean attribute identifying the riding winner.

#### House of Commons Members

| Attribute | Munzner Attribute Type | Ordering |
|------|-----------|-----|
| Person ID | Categorical (identifier) | Unordered |
| Honorific Title | Categorical | Unordered |
| First Name, Last Name | Categorical | Unordered |
| Constituency | Categorical | Unordered |
| Province / Territory | Categorical | Unordered |
| Political Affiliation | Categorical | Unordered |
| Start Date | Ordered (temporal) | Sequential |
| End Date | Ordered (temporal) | Sequential |

#### Statistics Canada GDP by Industry

| Attribute | Munzner Attribute Type | Ordering |
|------|-----------|-----|
| REF_DATE (year) | Ordered (temporal) | Sequential (1997-2024) |
| GEO (province/territory) | Categorical | Unordered |
| NAICS (industry classification) | Categorical / **Hierarchical** | Unordered within level, but parent-child tree structure across levels |
| Prices (current vs. chained dollars) | Categorical | Unordered (two measurement bases) |
| UOM (unit of measure) | Categorical | Unordered |
| SCALAR_FACTOR | Categorical | Unordered |
| VALUE (GDP in millions $) | Quantitative | Sequential (ratio scale) |

**Observations on NAICS hierarchy:** The industry codes form a tree. For example:
- Level 0: `All industries [T001]`
  - Level 1: `Goods-producing industries [T002]`, `Services-producing industries [T003]`
    - Level 2: `Mining, quarrying, and oil and gas extraction [21]`
      - Level 3: `Oil and gas extraction [211]`
        - Level 4: `Oil sands extraction [21114]`, `Oil and gas extraction (except oil sands) [21111]`

This hierarchy is critical for our analysis: we must select the appropriate level to avoid double-counting (e.g., using `[211]` rather than summing its children `[21111]` and `[21114]`).

In [54]:
# Extract party affiliation from the Candidate field
# The candidate column contains strings like "Paul Connors Liberal/Libéral"
# and "Adam Chambers ** Conservative/Conservateur" (** marks incumbents).
# We match known bilingual party suffixes from the end of the string. --important

party_patterns = {
    'Liberal/Libéral': 'Liberal',
    'Conservative/Conservateur': 'Conservative',
    'NDP-New Democratic Party/NPD-Nouveau Parti démocratique': 'NDP',
    'Bloc Québécois/Bloc Québécois': 'Bloc Québécois',
    'Green Party/Parti Vert': 'Green',
    "People's Party - PPC/Parti populaire - PPC": "PPC",
    'Independent/Indépendant(e)': 'Independent',
    'Parti Rhinocéros Party/Parti Rhinocéros Party': 'Rhinocéros',
    "Christian Heritage Party/Parti de l'Héritage Chrétien": 'CHP',
    'Marxist-Leninist/Marxiste-Léniniste': 'ML',
    'Communist/Communiste': 'Communist',
    'Libertarian/Libertarien': 'Libertarian',
    'Animal Protection Party/Parti Protection Animaux': 'Animal Protection',
    'Marijuana Party/Parti Marijuana': 'Marijuana',
    'No Affiliation/Aucune appartenance': 'No Affiliation',
    'Centrist/Centriste': 'Centrist',
    'United Party of Canada (UP)/Parti Uni du Canada (UP)': 'United',
    'Canadian Future Party/Parti Avenir Canadien': 'Canadian Future',
}


def extract_party(candidate_str):
    """Extract party name from the end of the candidate string."""
    c = candidate_str.replace('**', '').strip()
    for pattern, short_name in party_patterns.items():
        if c.endswith(pattern):
            return short_name
    return 'Other'


elections['Party'] = elections['Candidate/Candidat'].apply(extract_party)
elections['Is_Incumbent'] = elections['Candidate/Candidat'].str.contains(
    r'\*\*', na=False)

# Normalize province names: extract English portion before "/"
elections['Province_Clean'] = elections['Province'].str.split(
    '/').str[0].str.strip()

# Verify: check for any 'Other' parties that slipped through
other_count = (elections['Party'] == 'Other').sum()
print(f"Unmatched party entries: {other_count}")
print(f"\nParty distribution in elections data:")
print(elections['Party'].value_counts().to_string())

Unmatched party entries: 0

Party distribution in elections data:
Party
Liberal              342
Conservative         342
NDP                  342
PPC                  247
Green                232
Independent          159
Bloc Québécois        78
ML                    35
CHP                   32
Rhinocéros            29
Communist             24
Canadian Future       19
Centrist              19
No Affiliation        18
Libertarian           16
United                16
Animal Protection      7
Marijuana              2


In [55]:
# Build province-level aggregations

# Identify winners: rows with non-null Majority
winners = elections[elections['Majority/Majorité'].notna()].copy()
print(f"Total riding winners identified: {winners.shape[0]}")

# Seats won per party per province
seats_by_province = winners.groupby(
    ['Province_Clean', 'Party']).size().unstack(fill_value=0)

# Total votes per party per province
votes_by_province = elections.groupby(['Province_Clean', 'Party'])[
    'Votes Obtained/Votes obtenus'].sum().unstack(fill_value=0)

# Vote share per party per province
vote_totals = votes_by_province.sum(axis=1)
vote_share = votes_by_province.div(vote_totals, axis=0)

# Conservative vote share per province
con_vote_share = vote_share['Conservative'] if 'Conservative' in vote_share.columns else pd.Series(
    0, index=vote_share.index)

print(f"\nConservative vote share by province:")
print(con_vote_share.sort_values(ascending=False).to_string())

Total riding winners identified: 343

Conservative vote share by province:
Province_Clean
Saskatchewan                 0.645561
Alberta                      0.636060
Manitoba                     0.464107
Ontario                      0.438142
British Columbia             0.411735
New Brunswick                0.407093
Newfoundland and Labrador    0.396605
Yukon                        0.385183
Prince Edward Island         0.368631
Nova Scotia                  0.352698
Northwest Territories        0.333132
Nunavut                      0.260154
Quebec                       0.232848


In [56]:
# Filter and prepare StatCan GDP data

NAICS_COL = 'North American Industry Classification System (NAICS)'

# Find the most recent year with actual (non-null) data
all_industries = statcan[statcan[NAICS_COL] == 'All industries [T001]']
for yr in range(statcan['REF_DATE'].max(), 1996, -1):
    yr_data = all_industries[(all_industries['REF_DATE'] == yr) & (
        all_industries['Prices'] == 'Current dollars')]
    if yr_data['VALUE'].notna().sum() >= 10:
        most_recent_year = yr
        break

print(f"Most recent year with complete GDP data: {most_recent_year}")

statcan_recent = statcan[
    (statcan['REF_DATE'] == most_recent_year) &
    (statcan['Prices'] == 'Current dollars')
].copy()

# Key NAICS categories for our analysis
naics_keys = {
    'All industries [T001]': 'GDP_Total',
    'Goods-producing industries [T002]': 'GDP_Goods',
    'Services-producing industries [T003]': 'GDP_Services',
    'Oil and gas extraction [211]': 'GDP_OilGas',
    'Mining, quarrying, and oil and gas extraction [21]': 'GDP_MiningTotal',
    'Mining and quarrying (except oil and gas) [212]': 'GDP_MiningOther',
}

# Pivot to get one row per province with key industry columns
gdp_data = {}
for naics_name, col_name in naics_keys.items():
    subset = statcan_recent[statcan_recent[NAICS_COL]
                            == naics_name][['GEO', 'VALUE']].copy()
    subset = subset.rename(columns={'VALUE': col_name})
    subset = subset.set_index('GEO')
    gdp_data[col_name] = subset[col_name]

gdp_province = pd.DataFrame(gdp_data)
gdp_province.index.name = 'Province'

# Compute derived ratios
gdp_province['Goods_Services_Ratio'] = gdp_province['GDP_Goods'] / \
    gdp_province['GDP_Services']
gdp_province['OilGas_Pct'] = (
    gdp_province['GDP_OilGas'] / gdp_province['GDP_Total']) * 100
gdp_province['Mining_Pct'] = (
    gdp_province['GDP_MiningTotal'] / gdp_province['GDP_Total']) * 100
gdp_province['OtherMining_Pct'] = (
    gdp_province['GDP_MiningOther'] / gdp_province['GDP_Total']) * 100
gdp_province['OtherGoods_Pct'] = (
    (gdp_province['GDP_Goods'] - gdp_province['GDP_MiningTotal']) / gdp_province['GDP_Total']) * 100
gdp_province['Services_Pct'] = (
    gdp_province['GDP_Services'] / gdp_province['GDP_Total']) * 100

# Fill NaN with 0 for provinces/territories missing some industries
gdp_province = gdp_province.fillna(0)

print(
    f"\nGDP data available for {gdp_province.shape[0]} provinces/territories")
print(f"\nOil & Gas as % of GDP (top 5):")
print(gdp_province['OilGas_Pct'].sort_values(
    ascending=False).head().to_string())

Most recent year with complete GDP data: 2022

GDP data available for 13 provinces/territories

Oil & Gas as % of GDP (top 5):
Province
Alberta                      28.401743
Newfoundland and Labrador    25.172368
Saskatchewan                 13.802210
Northwest Territories         4.532318
British Columbia              4.431714


In [57]:
# Merge economic and political data into a single province-level dataframe

# Province abbreviations for labeling
prov_abbrev = {
    'Alberta': 'AB', 'British Columbia': 'BC', 'Manitoba': 'MB',
    'New Brunswick': 'NB', 'Newfoundland and Labrador': 'NL',
    'Northwest Territories': 'NT', 'Nova Scotia': 'NS', 'Nunavut': 'NU',
    'Ontario': 'ON', 'Prince Edward Island': 'PE',
    'Quebec': 'QC', 'Saskatchewan': 'SK', 'Yukon': 'YT'
}

# The major parties we track for seat composition
major_parties = ['Conservative', 'Liberal', 'NDP', 'Bloc Québécois', 'Green']

# Build the merged dataframe
merged = gdp_province.copy()
merged['Con_VoteShare'] = con_vote_share
merged['Lib_VoteShare'] = vote_share.get('Liberal', 0)
merged['NDP_VoteShare'] = vote_share.get('NDP', 0)
merged['Bloc_VoteShare'] = vote_share.get('Bloc Québécois', 0)
merged['Green_VoteShare'] = vote_share.get('Green', 0)

# Add seat counts for each major party
for party in major_parties:
    col = f'{party}_Seats'
    if party in seats_by_province.columns:
        merged[col] = seats_by_province[party]
    else:
        merged[col] = 0

merged['Total_Seats'] = merged[[
    f'{p}_Seats' for p in major_parties]].sum(axis=1)
# Add any seats from minor parties
all_seats = winners.groupby('Province_Clean').size()
merged['Total_Seats'] = all_seats

merged['Abbreviation'] = merged.index.map(prov_abbrev)

# Fill NaN values (territories may have missing GDP categories)
merged = merged.fillna(0)

print("Merged province-level dataset:")
print(merged[['Goods_Services_Ratio', 'OilGas_Pct',
      'Con_VoteShare', 'Total_Seats', 'Abbreviation']].to_string())

Merged province-level dataset:
                           Goods_Services_Ratio  OilGas_Pct  Con_VoteShare  Total_Seats Abbreviation
Province                                                                                            
Newfoundland and Labrador              0.954622   25.172368       0.396605            7           NL
Prince Edward Island                   0.395341    0.000000       0.368631            4           PE
Nova Scotia                            0.240081    0.000000       0.352698           11           NS
New Brunswick                          0.340561    0.013361       0.407093           10           NB
Quebec                                 0.378523    0.002418       0.232848           78           QC
Ontario                                0.293762    0.077851       0.438142          122           ON
Manitoba                               0.395289    1.768180       0.464107           14           MB
Saskatchewan                           1.179236   13.802210 

# Preliminary Exploration

Before designing our visualizations, we explore the basic structure of the data through summary statistics and simple plots. This helps us understand the distributions and identify initial patterns.

In [69]:
# Preliminary Exploration: National seat distribution & GDP by province

party_colors = {
    'Conservative': 'blue', 'Liberal': 'red', 'NDP': 'orange',
    'Bloc Québécois': 'lightblue', 'Green': 'green', 'Independent': 'gray'
}

# National seat counts from commons data
national_seats = commons['Political Affiliation'].value_counts().reset_index()
national_seats.columns = ['Party', 'Seats']
national_seats['Color'] = national_seats['Party'].map(
    party_colors).fillna('lightgray')

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'Seats in the House of Commons (2025 Election)',
    f'Total GDP by Province ({most_recent_year}, Current Dollars)'
], horizontal_spacing=0.2)

# Plot 1: National seat distribution
fig.add_trace(
    go.Bar(y=national_seats['Party'], x=national_seats['Seats'],
           orientation='h', marker_color=national_seats['Color'],
           text=national_seats['Seats'], textposition='outside',
           showlegend=False),
    row=1, col=1
)

# Plot 2: Total GDP by province
gdp_sorted = gdp_province['GDP_Total'].dropna(
).sort_values(ascending=True).reset_index()
gdp_sorted.columns = ['Province', 'GDP']
fig.add_trace(
    go.Bar(y=gdp_sorted['Province'], x=gdp_sorted['GDP'] / 1000,
           orientation='h', marker_color='darkblue',
           showlegend=False),
    row=1, col=2
)

fig.update_layout(height=500, width=1100,
                  title_text='Preliminary Exploration: Political & Economic Overview')
fig.update_xaxes(title_text='Number of Seats', row=1, col=1)
fig.update_xaxes(title_text='GDP (Billions $)', row=1, col=2)
fig.show()

In [74]:
# -- Preliminary Exploration: Oil/Gas GDP share and vote share by province --

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'Oil & Gas Sector Significance by Province',
    'Conservative Party Vote Share by Province (2025)'
], horizontal_spacing=0.2)

# Plot 3: Oil & Gas as % of GDP
oilgas_sorted = merged['OilGas_Pct'].sort_values(ascending=True)
colors_oil = ['black' if pct >
              5 else 'lightgray' for pct in oilgas_sorted.values]
fig.add_trace(
    go.Bar(y=oilgas_sorted.index, x=oilgas_sorted.values,
           orientation='h', marker_color=colors_oil, showlegend=False),
    row=1, col=1
)

# Plot 4: Conservative vote share
con_sorted = merged['Con_VoteShare'].sort_values(ascending=True)
fig.add_trace(
    go.Bar(y=con_sorted.index, x=con_sorted.values * 100,
           orientation='h', marker_color='blue', showlegend=False),
    row=1, col=2
)

fig.update_xaxes(title_text='Oil & Gas as % of Provincial GDP', row=1, col=1)
fig.update_xaxes(title_text='Conservative Vote Share (%)', row=1, col=2)
fig.update_layout(height=500, width=1200,
                  title_text='Preliminary Exploration: Resource Dependency & Conservative Support')
fig.show()

# Summary statistics
print("\nKey statistics for merged province-level data:")
print(merged[['Goods_Services_Ratio', 'OilGas_Pct',
      'Con_VoteShare', 'Total_Seats']].describe().round(3).to_string())


Key statistics for merged province-level data:
       Goods_Services_Ratio  OilGas_Pct  Con_VoteShare  Total_Seats
count                13.000      13.000         13.000       13.000
mean                  0.559       6.016          0.410       26.385
std                   0.317      10.015          0.121       36.306
min                   0.240       0.000          0.233        1.000
25%                   0.341       0.000          0.353        4.000
50%                   0.395       0.078          0.397       11.000
75%                   0.868       4.532          0.438       37.000
max                   1.179      28.402          0.646      122.000


# Purpose of Visualizations - Munzner "Why" Analysis

We now perform a systematic *why* analysis following Munzner's action-target pair taxonomy across three levels of analysis: **Analyze** (high-level goals), **Search** (what is known vs. unknown), and **Query** (what output is needed).

### Question 1: "Is there a relationship between a province's economics and its party support?"

This is an **exploratory** question - we do not know in advance whether a relationship exists, and if so, what form it takes.

#### Analyze Level (High-Level Goals)

| Action | Target | Justification |
|----|----|--------|
| **Discover** | **Trends** | We want the visualization to *reveal* whether provinces with similar economic structures tend to support the same parties. Since this is exploratory, "discover" is the appropriate action - we are not presenting a known fact. The target is "trends" because we are looking for a systematic pattern across multiple provinces, not a single data point. |
| **Derive** | **Correlation** | We want to derive whether there is a quantitative correlation between economic indicators (e.g., goods-to-services GDP ratio) and party vote share. Both are ratio-scale quantitative attributes, making correlation a natural derived quantity. |

#### Search Level (Known vs. Unknown)

| Action | Target | Justification |
|----|----|--------|
| **Explore** | **Distribution** | The identities of the provinces are known, but their positions in the joint economy-politics space are unknown. We want to explore how 13 provinces/territories distribute across the two-dimensional space of economic structure vs. political outcome. "Explore" is appropriate because both the location and the identity of interesting patterns are unknown. |
| **Lookup** | **Values** | For specific provinces of interest (e.g., Ontario as the largest, Alberta as the most resource-dependent), we want to look up their exact economic profile and corresponding party vote shares. Here the target province is known, and we seek its specific attribute values. |

#### Query Level (Output Needed)

| Action | Target | Justification |
|----|----|--------|
| **Compare** | **Features** | We need to compare the economic profiles and voting outcomes across all provinces simultaneously. "Compare" requires showing multiple items side by side; "features" refers to the multiple attributes (GDP ratio, vote shares, seat counts) of each province. |
| **Identify** | **Outliers** | We want to identify provinces that deviate from the general trend - for example, a province with a resource-heavy economy that does not vote Conservative, or a service-oriented province that does. Outliers are the most informative data points for understanding the limits of the relationship. |

### Question 2: "Is there a relationship between Alberta's conservative orientation and its industry, particularly its oil industry, as compared to other provinces?"

This is a more **hypothesis-driven** question - we are testing the specific claim that Alberta's oil-dependent economy explains its strong Conservative support, and comparing it to other provinces.

#### Analyze Level (High-Level Goals)

| Action | Target | Justification |
|----|----|--------|
| **Present** | **Comparison** | Unlike Q1, this question has a hypothesis to test (Alberta's oil drives its conservatism). The action is "present" rather than "discover" because we are arranging the data to evaluate a specific claim. The target is "comparison" because the question explicitly asks about Alberta *relative to* other provinces. |
| **Discover** | **Similarity / Difference** | Beyond Alberta, we want to discover which provinces are most similar to or different from Alberta in both economic structure and political alignment. Saskatchewan (also oil/gas-dependent) and Newfoundland (has offshore oil but votes Liberal) are expected to be informative contrasts. |

#### Search Level (Known vs. Unknown)

| Action | Target | Justification |
|----|----|--------|
| **Locate** | **Known Item** | Alberta is a specific, known focal entity. We need to locate it quickly within the visualization so the viewer can immediately begin the comparison. Other oil-producing provinces (Saskatchewan, Newfoundland) are also known targets to locate. |
| **Browse** | **Features** | We want to browse the industry breakdown (oil/gas, other mining, other goods, services) of each province to understand what makes Alberta's economy structurally distinct. "Browse" is appropriate because we are scanning across multiple feature values for multiple items without a precise search target. |

#### Query Level (Output Needed)

| Action | Target | Justification |
|----|----|--------|
| **Compare** | **Feature Values** | We need to quantitatively compare Alberta's oil/gas GDP share and Conservative vote share against every other province. This is a direct comparison of specific quantitative attribute values across items. |
| **Summarize** | **Distribution** | We want to summarize the overall relationship between oil/resource sector proportion and Conservative vote share across all provinces, with Alberta as the focal point. "Summarize" captures the need for an overview pattern, not just individual comparisons. |

# Design of Visuals - Munzner "How" Analysis

### Visualization 1: "Economic-Political Landscape" - Pie-Glyph Scatter Plot

#### Connection to Action-Target Pairs
This visualization addresses Q1's action-target pairs:
- **Discover / Trends**: The scatter layout reveals whether provinces with higher goods-to-services ratios tend to have higher Conservative vote shares, showing (or not) a trend.
- **Explore / Distribution**: Each province's position in the 2D space of economy vs. politics is shown simultaneously.
- **Compare / Features**: All provinces are plotted together, enabling direct spatial comparison.
- **Identify / Outliers**: Provinces far from the trend line are immediately visible as outliers.
- Also partially supports Q2's **Locate / Known Item** (Alberta is labeled and identifiable).

#### Idiom Description
- **Base idiom**: Scatter plot - the most effective idiom for showing the relationship between two quantitative variables (Munzner Ch. 7).
- **X-axis**: Goods-to-Services GDP ratio (quantitative, ratio scale) - captures how "resource/manufacturing-heavy" vs. "service-oriented" a province's economy is.
- **Y-axis**: Conservative vote share (quantitative, ratio scale, range 0-1).
- **Mark**: Each province is represented by a **miniature pie-chart glyph** at its scatter position. The pie slices show the **seat distribution** among major parties.
- **Size channel**: Pie glyph radius encodes the total number of seats (a proxy for population), making Ontario and Quebec visually larger.
- **Annotation**: Province abbreviation labels and a linear regression trend line.

#### Channel Justification (from What/Why)
- **Position** (horizontal and vertical): Munzner ranks position as the most effective channel for quantitative data. We use it for the two primary quantitative variables.
- **Color hue**: Effective for unordered categorical data (Munzner Ch. 5). Distinguishes party affiliations within each pie glyph.
- **Size / area**: Encodes the quantitative "total seats" attribute. Less precise than position but adds a third quantitative dimension without requiring a separate panel.
- **Angle (pie slices)**: Encodes part-to-whole seat composition within each province. While pie charts have known perceptual limitations for precise angle comparison, at the province level (3-5 parties) they are readable and serve the Compare/Features goal.

#### Sources and Originality
- **Inspiration**: Hans Rosling's Gapminder bubble charts (size-encoded scatter plots); epidemiological pie-on-map visualizations that place pie charts at geographic locations to show composition; Munzner Ch. 7 on glyphs as complex multi-attribute marks.
- **Original element**: The combination of pie-chart glyphs within a non-geographic scatter plot is the original design contribution. Standard scatter plots use simple point marks; by replacing each point with a miniature pie chart, we encode three additional dimensions (party seat breakdown) per mark without requiring a separate visualization. The dual encoding of Conservative vote *share* (Y-axis position) and seat *composition* (pie glyph) also allows viewers to see whether vote share translates proportionally into seats, revealing electoral system effects.

### Visualization 2: "Alberta in Context" - Dual-Panel Industry Composition + Lollipop Chart

#### Connection to Action-Target Pairs
This visualization addresses Q2's action-target pairs:
- **Present / Comparison**: Alberta's industry profile is directly juxtaposed with all other provinces.
- **Locate / Known Item**: Alberta is highlighted with a yellow background band, enabling instant identification (pre-attentive pop-out).
- **Browse / Features**: The stacked bar shows the full industry breakdown for every province.
- **Compare / Feature Values**: The lollipop panel enables precise quantitative comparison of Conservative vote share.
- **Summarize / Distribution**: Sorting provinces by Conservative vote share creates an implicit ranking that reveals whether oil-heavy economies cluster at the top.

#### Idiom Description
- **Layout**: Two horizontally adjacent panels sharing a vertical axis (provinces listed vertically).
- **Left panel - Normalized stacked horizontal bar**: Each province's GDP is decomposed into four industry categories (Oil & Gas, Other Mining/Resources, Other Goods-producing, Services-producing), normalized to 100% width. This shows *composition*, not absolute size.
- **Right panel - Lollipop chart**: A dot-and-stem plot showing Conservative vote share (0-100%) for each province. A vertical reference line marks the national average.
- **Sorting**: Provinces are sorted by Conservative vote share (descending), so the most Conservative provinces appear at the top.
- **Alberta highlight**: A pale yellow horizontal band spans the full width at Alberta's row, using pre-attentive color pop-out to draw immediate attention.

#### Channel Justification (from What/Why)
- **Vertical position** (shared axis, ordered by vote share): Enables the Compare and Summarize actions. Sorting by the political outcome variable lets the viewer scan top-to-bottom and assess whether industry composition co-varies with political alignment.
- **Length** (horizontal bar segments): Munzner ranks length as highly effective for quantitative data. Used for the GDP composition proportions.
- **Color hue** (industry categories): Distinguishes the four economic sectors (categorical, unordered).
- **Horizontal position** (lollipop dot): Encodes Conservative vote share precisely.
- **Color saturation / luminance** (Alberta highlight band): Pre-attentive pop-out draws attention to the focal province without requiring visual search, supporting the Locate/Known Item action.

In [79]:
# ============================================================
# VISUALIZATION 1: Pie-Glyph Scatter Plot
# "Economic-Political Landscape"
# ============================================================

import base64
from io import BytesIO
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

pie_party_colors = {
    'Conservative': 'blue',
    'Liberal': 'red',
    'NDP': 'orange',
    'Bloc Québécois': 'lightblue',
    'Green': 'green',
}


def make_pie_image(seat_values, seat_labels, colors, size_px=120):
    """Create a small pie chart as a base64-encoded PNG."""
    fig_pie, ax_pie = plt.subplots(figsize=(1.2, 1.2), dpi=100)
    # Filter out zero-seat parties
    nonzero = [(v, l, c)
               for v, l, c in zip(seat_values, seat_labels, colors) if v > 0]
    if not nonzero:
        plt.close(fig_pie)
        return None
    vals, labs, cols = zip(*nonzero)
    ax_pie.pie(vals, colors=cols, startangle=90, counterclock=False,
               wedgeprops={'edgecolor': 'white', 'linewidth': 1})
    ax_pie.set_aspect('equal')
    buf = BytesIO()
    fig_pie.savefig(buf, format='png', transparent=True,
                    bbox_inches='tight', pad_inches=0.02)
    plt.close(fig_pie)
    buf.seek(0)
    return 'data:image/png;base64,' + base64.b64encode(buf.read()).decode()


# Prepare data
valid = merged.dropna(subset=['Goods_Services_Ratio', 'Con_VoteShare']).copy()
valid = valid[valid['Total_Seats'] > 0]

# Compute trend line
x_arr = valid['Goods_Services_Ratio'].values.astype(float)
y_arr = valid['Con_VoteShare'].values.astype(float)

# Remove any inf/nan
mask = np.isfinite(x_arr) & np.isfinite(y_arr)
x_clean, y_clean = x_arr[mask], y_arr[mask]

if len(x_clean) > 2:
    z = np.polyfit(x_clean, y_clean, 1)
    p_func = np.poly1d(z)
    r_val = np.corrcoef(x_clean, y_clean)[0, 1]
    x_line = np.linspace(x_clean.min() - 0.02, x_clean.max() + 0.05, 100)
    y_line = p_func(x_line)

In [ ]:
# Build the Plotly figure
fig = go.Figure()

# Add trend line
if len(x_clean) > 2:
    fig.add_trace(go.Scatter(
        x=x_line, y=y_line, mode='lines',
        line=dict(color="black", width=2, dash='dash'),
        name=f'Trend (r={r_val:.2f})', hoverinfo='skip'
    ))

# Add invisible scatter points for hover information
hover_texts = []
for idx, row in valid.iterrows():
    parts = []
    parts.append(f"<b>{idx}</b> ({row['Abbreviation']})")
    parts.append(f"Goods/Services Ratio: {row['Goods_Services_Ratio']:.3f}")
    parts.append(f"Conservative Vote Share: {row['Con_VoteShare']*100:.1f}%")
    parts.append(f"Total Seats: {int(row['Total_Seats'])}")
    for party in major_parties:
        seats = int(row.get(f'{party}_Seats', 0))
        if seats > 0:
            parts.append(f"  {party}: {seats} seats")
    hover_texts.append('<br>'.join(parts))

fig.add_trace(go.Scatter(
    x=valid['Goods_Services_Ratio'], y=valid['Con_VoteShare'],
    mode='markers+text',
    marker=dict(size=1, color='black'),
    text=valid['Abbreviation'],
    textposition='top right',
    textfont=dict(size=10),
    hovertext=hover_texts, hoverinfo='text',
    showlegend=False
))

# Add pie chart images as layout images
x_range = [x_clean.min() - 0.05, x_clean.max() + 0.1]
y_range = [0, y_clean.max() + 0.12]

images = []
for idx, row in valid.iterrows():
    seat_values = [row.get(f'{p}_Seats', 0) for p in major_parties]
    other = row['Total_Seats'] - sum(seat_values)
    if other > 0:
        seat_values.append(other)
        colors = list(pie_party_colors.values()) + ['gray']
        labels = major_parties + ['Other']
    else:
        colors = list(pie_party_colors.values())
        labels = major_parties

    img_data = make_pie_image(seat_values, labels, colors)
    if img_data is None:
        continue

    # Scale size by sqrt of total seats
    max_seats = valid['Total_Seats'].max()
    size_frac = 0.04 + 0.10 * \
        (np.sqrt(row['Total_Seats']) / np.sqrt(max_seats))

    # Convert data coords to fraction of plot area
    xf = (row['Goods_Services_Ratio'] - x_range[0]) / (x_range[1] - x_range[0])
    yf = (row['Con_VoteShare'] - y_range[0]) / (y_range[1] - y_range[0])

    # Map to paper coordinates
    margin_l, margin_r = 0.08, 0.98
    margin_b, margin_t = 0.10, 0.88
    x_paper = margin_l + xf * (margin_r - margin_l)
    y_paper = margin_b + yf * (margin_t - margin_b)

    images.append(dict(
        source=img_data,
        xref='paper', yref='paper',
        x=x_paper, y=y_paper,
        sizex=size_frac, sizey=size_frac,
        xanchor='center', yanchor='middle',
        layer='above'
    ))

# Add legend entries for party colors
for party, color in pie_party_colors.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=12, color=color),
        name=party, showlegend=True
    ))

fig.update_layout(
    images=images,
    xaxis=dict(title='Goods-to-Services GDP Ratio',
               range=x_range, gridcolor='#EEEEEE'),
    yaxis=dict(title='Conservative Vote Share',
               range=y_range, gridcolor='#EEEEEE'),
    title=dict(
        text='Economic-Political Landscape: Provincial Economy vs. Conservative Support<br>'
             '<sup>Pie glyphs show party seat composition; size encodes total seats</sup>',
        font=dict(size=15)
    ),
    plot_bgcolor='white',
    width=900, height=650,
    legend=dict(x=0.75, y=0.05, bordercolor='lightgray', borderwidth=1)
)

fig.show()

In [ ]:
# ============================================================
# VISUALIZATION 2: Dual-Panel Industry Composition + Lollipop
# "Alberta in Context"
# ============================================================

# Sort provinces by Conservative vote share (descending top-to-bottom in plotly = ascending in data)
sorted_merged = merged.sort_values('Con_VoteShare', ascending=True).copy()
provinces = sorted_merged.index.tolist()
n = len(provinces)

# Handle NaN for territories
for col in ['OilGas_Pct', 'OtherMining_Pct', 'OtherGoods_Pct', 'Services_Pct']:
    sorted_merged[col] = sorted_merged[col].fillna(0)

# Normalize to 100%
industry_cols = ['OilGas_Pct', 'OtherMining_Pct',
                 'OtherGoods_Pct', 'Services_Pct']
row_totals = sorted_merged[industry_cols].sum(axis=1)
for col in industry_cols:
    sorted_merged[col + '_norm'] = sorted_merged[col] / row_totals * 100

# Province labels
prov_labels = [
    f"{p} ({sorted_merged.loc[p, 'Abbreviation']})" for p in provinces]

# Industry styling
industry_config = [
    ('OilGas_Pct_norm', 'Oil & Gas', 'black'),
    ('OtherMining_Pct_norm', 'Other Mining & Resources', 'gray'),
    ('OtherGoods_Pct_norm', 'Other Goods-Producing', 'darkblue'),
    ('Services_Pct_norm', 'Services-Producing', 'red'),
]

# National average Conservative vote share
national_avg = elections[elections['Party'] == 'Conservative']['Votes Obtained/Votes obtenus'].sum() / \
    elections['Votes Obtained/Votes obtenus'].sum() * 100

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, column_widths=[0.6, 0.4],
                    horizontal_spacing=0.02,
                    subplot_titles=['Industry Composition of Provincial GDP',
                                    'Conservative Vote Share (%)'])

# Left Panel: Stacked horizontal bars
for col_key, label, color in industry_config:
    fig.add_trace(
        go.Bar(
            y=prov_labels, x=sorted_merged[col_key].values,
            name=label, marker_color=color, orientation='h',
            legendgroup=label, showlegend=True,
            hovertemplate=f'{label}: ' + '%{x:.1f}%<extra></extra>'
        ),
        row=1, col=1
    )

# Right Panel: Lollipop chart
con_values = sorted_merged['Con_VoteShare'].values * 100

# National average reference line
fig.add_trace(
    go.Scatter(
        x=[national_avg, national_avg], y=[prov_labels[0], prov_labels[-1]],
        mode='lines', line=dict(color='gray', width=1.5, dash='dash'),
        name=f'National avg ({national_avg:.1f}%)', showlegend=True,
        hoverinfo='skip'
    ),
    row=1, col=2
)

# Lollipop stems and dots
for i, (prov, label, val) in enumerate(zip(provinces, prov_labels, con_values)):
    is_alberta = (prov == 'Alberta')
    fig.add_trace(
        go.Scatter(
            x=[0, val], y=[label, label],
            mode='lines',
            line=dict(color='blue', width=3 if is_alberta else 1.5),
            showlegend=False, hoverinfo='skip'
        ),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter(
            x=[val], y=[label],
            mode='markers+text',
            marker=dict(
                size=14 if is_alberta else 9,
                color='blue',
                line=dict(color='white', width=2) if is_alberta else dict(
                    width=0)
            ),
            text=[f'{val:.1f}%'], textposition='middle right',
            textfont=dict(size=9, color='black'),
            showlegend=False,
            hovertemplate=f'<b>{prov}</b><br>Conservative: {val:.1f}%<extra></extra>'
        ),
        row=1, col=2
    )

# Alberta highlight band
alberta_idx = provinces.index('Alberta') if 'Alberta' in provinces else None
if alberta_idx is not None:
    for col_idx in [1, 2]:
        fig.add_shape(
            type='rect',
            x0=0, x1=1, xref=f'x{col_idx} domain' if col_idx > 1 else 'x domain',
            y0=alberta_idx - 0.4, y1=alberta_idx + 0.4,
            yref='y' if col_idx == 1 else 'y2',
            fillcolor='rgba(255, 250, 205, 0.6)', line_width=0,
            layer='below',
            row=1, col=col_idx
        )

fig.update_layout(
    barmode='stack',
    height=600, width=1100,
    title=dict(
        text="Alberta in Context: Industry Structure vs. Conservative Support<br>"
             "<sup>Sorted by Conservative vote share; Alberta highlighted in yellow</sup>",
        font=dict(size=15)
    ),
    plot_bgcolor='white',
    legend=dict(x=0.01, y=-0.15, orientation='h', font=dict(size=9))
)

fig.update_xaxes(title_text='GDP Composition (%)',
                 range=[0, 100], row=1, col=1)
fig.update_xaxes(title_text='Vote Share (%)', range=[
                 0, max(con_values) + 12], row=1, col=2)

fig.show()

# Conclusion

### Visualization 1: Pie-Glyph Scatter Plot - Evaluation

**Addressing the action-target pairs for Q1 ("Is there a relationship between a province's economics and its party support?"):**

- **Discover / Trends**: The scatter plot, together with the trend line and correlation coefficient, reveals whether a positive relationship exists between goods-to-services GDP ratio and Conservative vote share. Provinces that are more goods-producing-oriented (higher ratio) tend to appear higher on the y-axis, suggesting a trend - though the small sample size (13 provinces/territories) limits statistical confidence.
- **Explore / Distribution**: The scatter layout successfully shows how provinces distribute across the economy-politics space. Clustering is visible: prairie provinces (AB, SK) occupy the upper-right (goods-heavy, high Conservative share), while central/Atlantic provinces tend toward the lower-left.
- **Compare / Features**: The pie glyphs allow simultaneous comparison of both economic positioning (via scatter position) and political composition (via pie slices). For example, Quebec's pie shows mostly Bloc Québécois seats despite moderate economics, immediately revealing that provincial politics (sovereignty movement) confound the economic hypothesis.
- **Identify / Outliers**: Provinces that deviate from the trend line are visually apparent. Newfoundland, which has significant oil/gas extraction but votes Liberal, is a notable outlier that challenges a simple economic-determinism narrative.

**Answering Q1**: The visualization suggests a *partial* relationship. Goods-producing provinces do tend to lean Conservative, but the relationship is not deterministic. Quebec's Bloc support and Newfoundland's Liberal orientation despite oil resources demonstrate that cultural, historical, and regional factors also drive voting patterns.

**Limitations**: With only 13 data points, the trend line has limited statistical power. The pie glyphs, while informative, become very small for territories with 1 seat, making their internal composition hard to read.



### Visualization 2: Dual-Panel Industry + Lollipop - Evaluation

**Addressing the action-target pairs for Q2 ("Is there a relationship between Alberta's conservative orientation and its industry, particularly its oil industry, as compared to other provinces?"):**

- **Present / Comparison**: The side-by-side layout directly juxtaposes Alberta's industry composition with every other province, making the comparison immediate and visual.
- **Locate / Known Item**: The yellow highlight band makes Alberta instantly identifiable without visual search, successfully employing pre-attentive pop-out.
- **Browse / Features**: The stacked bars allow viewers to scan through each province's industry breakdown. Alberta's distinctively large Oil & Gas (black) segment stands out.
- **Compare / Feature Values**: The lollipop panel enables precise comparison: viewers can see Alberta's exact Conservative vote share relative to both the national average line and other provinces.
- **Summarize / Distribution**: The sort-by-Conservative-vote-share ordering reveals whether oil-heavy provinces cluster at the high end. Alberta and Saskatchewan, the two provinces with the largest oil/gas sectors, do indeed appear near the top.

**Answering Q2**: The visualization provides suggestive evidence that Alberta's oil industry is associated with its Conservative orientation. Alberta has by far the largest oil/gas GDP share and one of the highest Conservative vote shares. Saskatchewan, the other major oil-producing province, also shows high Conservative support. However, the relationship is not perfectly clean. Other provinces with minimal oil/gas also vote Conservative at above-average rates, and Newfoundland (which has some oil/gas) votes Liberal. This suggests oil dependence is one contributing factor among several.

### Suggestions for Improved Designs

1. **Geographic encoding for Viz 1**: Replace the abstract scatter axes with a geographic map of Canada, positioning pie glyphs at each province's centroid. This would add spatial context (e.g., the "Western alienation" narrative) while using color/size for economic and political variables. The trade-off is losing the precise quantitative positioning of the scatter axes.

2. **Temporal dimension for Viz 2**: Add a small-multiple panel showing how Alberta's oil GDP share and Conservative vote share have co-evolved across multiple elections (2006, 2008, 2011, 2015, 2019, 2021, 2025). This would strengthen causal claims by showing whether changes in oil dependence track changes in Conservative support.

# Attributions

| Source | What is it | How used |
|----|------|-----|
| [Elections Canada - Table 12](https://www.elections.ca/content.aspx?section=res&dir=rep/off/45gedata&document=summary&lang=e) | Official 45th General Election candidate-level voting results | Primary data source for riding-level votes, party support, and seat winners. Used throughout Sections 3-6. |
| [Statistics Canada - Table 36-10-0711-01](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=3610071101) | GDP at basic prices by industry, by province/territory | Primary data source for provincial economic structure (industry composition, oil/gas share). Used in Sections 3-6. |
| [House of Commons of Canada](https://www.ourcommons.ca/Members/en/search/csv) | Current Members of Parliament CSV download | Data source for clean party affiliation column and seat verification. Used in Sections 3-4. |
| Munzner, T. *Visualization Analysis and Design* (2014), CRC Press | Textbook on visualization design methodology | Framework for what/why/how analysis used throughout. Specific references to Ch. 2 (what), Ch. 3 (why: action-target pairs), Ch. 5 (channel effectiveness rankings), Ch. 7 (idioms, glyphs, juxtaposed views). |
| Hans Rosling / Gapminder Foundation | Bubble chart visualization design | Inspiration for size-encoded scatter plot concept in Visualization 1. Adapted by replacing simple bubble marks with pie-chart glyphs. |
| D3.js Observable Gallery - Lollipop Charts | Visualization idiom examples | Inspiration for the lollipop chart component of Visualization 2. The stem-and-dot design was chosen over standard bars for cleaner visual comparison. |
| matplotlib documentation (matplotlib.org) | Python plotting library reference | Implementation reference for `Wedge` patches (pie glyphs), `barh` (stacked bars), `hlines`/`scatter` (lollipop), and `axhspan` (highlight band). |
| ChatGPT | Assignment Visualization 1 | Plotly does not natively support pie-chart markers within scatter plots. I used ChatGPT to create  a main scatter plot with bubble markers colored by the dominant party, sized by total seats, combined with a grid of mini pie charts displayed as subplot insets positioned to match the scatter. |